# Abstractive without pca

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Load your dataset
file_path ="C:/Users/Shanmukha Reddy/Desktop/ML/t5_embeddings_abstractive.xlsx"  # Replace with the path to your dataset
dataset = pd.read_excel(file_path)

# Convert column names to strings to avoid errors
dataset.columns = dataset.columns.astype(str)

# Separate features (X) and target (y)
X = dataset.drop(columns=['Judgement Status'])
y = dataset['Judgement Status']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data (necessary for KNN and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define classifiers (with fixes for warnings)
classifiers = {
    'XGBoost': xgb.XGBClassifier(eval_metric='mlogloss'),  # Removed use_label_encoder
    'AdaBoost': AdaBoostClassifier(algorithm='SAMME'),  # Specified SAMME to avoid warning
    'CatBoost': CatBoostClassifier(verbose=0),
    'RandomForest': RandomForestClassifier(),
    'DecisionTree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC()
}

# Train and evaluate each model
metrics = {}

for model_name, model in classifiers.items():
    # Fit model on training data
    if model_name in ['KNN', 'SVM']:
        model.fit(X_train_scaled, y_train)
        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
    
    # Calculate train and test metrics
    metrics[model_name] = {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train F1-Score': f1_score(y_train, y_train_pred, average='weighted'),
        'Test F1-Score': f1_score(y_test, y_test_pred, average='weighted')
    }

# Display the metrics for each model
metrics_df = pd.DataFrame(metrics).T
print(metrics_df)


              Train Accuracy  Test Accuracy  Train F1-Score  Test F1-Score
XGBoost             1.000000       0.391667        1.000000       0.388066
AdaBoost            0.552083       0.391667        0.541120       0.372730
CatBoost            1.000000       0.350000        1.000000       0.328172
RandomForest        1.000000       0.375000        1.000000       0.339815
DecisionTree        1.000000       0.233333        1.000000       0.231982
KNN                 0.560417       0.350000        0.552246       0.331280
SVM                 0.927083       0.375000        0.927114       0.328032


# Abstractive with pca

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
import numpy as np

# Load your dataset
file_path = "C:/Users/Shanmukha Reddy/Desktop/ML/t5_embeddings_abstractive.xlsx"  # Replace with the path to your dataset
dataset = pd.read_excel(file_path)

# Convert column names to strings to avoid errors
dataset.columns = dataset.columns.astype(str)

# Separate features (X) and target (y)
X = dataset.drop(columns=['Judgement Status'])
y = dataset['Judgement Status']

# Apply SMOTE for class balancing
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply PCA (reduce dimensionality to 20 components)
pca = PCA(n_components=20)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Define classifiers with parameters to reduce overfitting
classifiers = {
    'XGBoost': xgb.XGBClassifier(n_estimators=200, learning_rate=0.01, max_depth=3, eval_metric='mlogloss', n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, learning_rate=0.01),
    'CatBoost': CatBoostClassifier(iterations=200, learning_rate=0.01, depth=3, verbose=0),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1),
    'DecisionTree': DecisionTreeClassifier(max_depth=10, min_samples_split=5, min_samples_leaf=2),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'SVM': SVC(kernel='rbf', C=1, gamma=0.01)
}

# Use Stratified K-Folds Cross-Validation
skf = StratifiedKFold(n_splits=5)
metrics = {}

for model_name, model in classifiers.items():
    model.fit(X_train_pca, y_train)
    
    # Predict on training and test data
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)
    
    # Calculate metrics
    metrics[model_name] = {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train Recall': recall_score(y_train, y_train_pred, average='weighted'),
        'Test Recall': recall_score(y_test, y_test_pred, average='weighted'),
        'Train F1-Score': f1_score(y_train, y_train_pred, average='weighted'),
        'Test F1-Score': f1_score(y_test, y_test_pred, average='weighted')
    }

# Display the metrics for each model
metrics_df = pd.DataFrame(metrics).T
print(metrics_df)


C:\Users\Shanmukha Reddy\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


              Train Accuracy  Test Accuracy  Train Recall  Test Recall  \
XGBoost             0.751332       0.453901      0.751332     0.453901   
AdaBoost            0.403197       0.354610      0.403197     0.354610   
CatBoost            0.637655       0.425532      0.637655     0.425532   
RandomForest        0.998224       0.524823      0.998224     0.524823   
DecisionTree        0.870337       0.397163      0.870337     0.397163   
KNN                 0.625222       0.453901      0.625222     0.453901   
SVM                 0.996448       0.574468      0.996448     0.574468   

              Train F1-Score  Test F1-Score  
XGBoost             0.752430       0.451405  
AdaBoost            0.399906       0.346758  
CatBoost            0.636641       0.420989  
RandomForest        0.998224       0.524315  
DecisionTree        0.870460       0.388994  
KNN                 0.621856       0.449742  
SVM                 0.996448       0.580070  


# Extractive without pca

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Load your dataset
file_path ="C:/Users/Shanmukha Reddy/Desktop/ML/t5_embeddings_extractive.xlsx"  # Replace with the path to your dataset
dataset = pd.read_excel(file_path)

# Convert column names to strings to avoid errors
dataset.columns = dataset.columns.astype(str)

# Separate features (X) and target (y)
X = dataset.drop(columns=['Judgement Status'])
y = dataset['Judgement Status']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data (necessary for KNN and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define classifiers (with fixes for warnings)
classifiers = {
    'XGBoost': xgb.XGBClassifier(eval_metric='mlogloss'),  # Removed use_label_encoder
    'AdaBoost': AdaBoostClassifier(algorithm='SAMME'),  # Specified SAMME to avoid warning
    'CatBoost': CatBoostClassifier(verbose=0),
    'RandomForest': RandomForestClassifier(),
    'DecisionTree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC()
}

# Train and evaluate each model
metrics = {}

for model_name, model in classifiers.items():
    # Fit model on training data
    if model_name in ['KNN', 'SVM']:
        model.fit(X_train_scaled, y_train)
        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
    
    # Calculate train and test metrics
    metrics[model_name] = {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train F1-Score': f1_score(y_train, y_train_pred, average='weighted'),
        'Test F1-Score': f1_score(y_test, y_test_pred, average='weighted')
    }

# Display the metrics for each model
metrics_df = pd.DataFrame(metrics).T
print(metrics_df)


              Train Accuracy  Test Accuracy  Train F1-Score  Test F1-Score
XGBoost             1.000000       0.441667        1.000000       0.427284
AdaBoost            0.579167       0.333333        0.576713       0.328993
CatBoost            1.000000       0.358333        1.000000       0.344494
RandomForest        1.000000       0.300000        1.000000       0.260583
DecisionTree        1.000000       0.258333        1.000000       0.245421
KNN                 0.597917       0.433333        0.596245       0.428012
SVM                 0.933333       0.400000        0.933049       0.373679


# Extractive with pca

In [7]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
import xgboost as xgb
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
import numpy as np

# Load your dataset
file_path = "C:/Users/Shanmukha Reddy/Desktop/ML/t5_embeddings_extractive.xlsx"   # Replace with the path to your dataset
dataset = pd.read_excel(file_path)

# Convert column names to strings to avoid errors
dataset.columns = dataset.columns.astype(str)

# Separate features (X) and target (y)
X = dataset.drop(columns=['Judgement Status'])
y = dataset['Judgement Status']

# Apply SMOTE for class balancing
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply PCA (reduce dimensionality to 20 components)
pca = PCA(n_components=20)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Define classifiers with parameters to reduce overfitting
classifiers = {
    'XGBoost': xgb.XGBClassifier(n_estimators=200, learning_rate=0.01, max_depth=3, eval_metric='mlogloss', n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, learning_rate=0.01),
    'CatBoost': CatBoostClassifier(iterations=200, learning_rate=0.01, depth=3, verbose=0),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, min_samples_leaf=2, n_jobs=-1),
    'DecisionTree': DecisionTreeClassifier(max_depth=10, min_samples_split=5, min_samples_leaf=2),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'SVM': SVC(kernel='rbf', C=1, gamma=0.01)
}

# Use Stratified K-Folds Cross-Validation
skf = StratifiedKFold(n_splits=5)
metrics = {}

for model_name, model in classifiers.items():
    model.fit(X_train_pca, y_train)
    
    # Predict on training and test data
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)
    
    # Calculate metrics
    metrics[model_name] = {
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Train Recall': recall_score(y_train, y_train_pred, average='weighted'),
        'Test Recall': recall_score(y_test, y_test_pred, average='weighted'),
        'Train F1-Score': f1_score(y_train, y_train_pred, average='weighted'),
        'Test F1-Score': f1_score(y_test, y_test_pred, average='weighted')
    }

# Display the metrics for each model
metrics_df = pd.DataFrame(metrics).T
print(metrics_df)


C:\Users\Shanmukha Reddy\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


              Train Accuracy  Test Accuracy  Train Recall  Test Recall  \
XGBoost             0.761989       0.460993      0.761989     0.460993   
AdaBoost            0.422735       0.347518      0.422735     0.347518   
CatBoost            0.603908       0.432624      0.603908     0.432624   
RandomForest        0.996448       0.581560      0.996448     0.581560   
DecisionTree        0.886323       0.411348      0.886323     0.411348   
KNN                 0.632327       0.453901      0.632327     0.453901   
SVM                 0.998224       0.560284      0.998224     0.560284   

              Train F1-Score  Test F1-Score  
XGBoost             0.760670       0.458133  
AdaBoost            0.424574       0.344509  
CatBoost            0.602026       0.429205  
RandomForest        0.996441       0.578792  
DecisionTree        0.887223       0.410714  
KNN                 0.628754       0.444264  
SVM                 0.998224       0.560037  
